In [13]:
import os
import time
import random
import json
import requests
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from bs4 import BeautifulSoup

from IPython.display import display
from pprint import pprint
import tqdm.auto as tqdm

from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed

In [14]:
CSV_IN = "ncbi_bacteria_links.csv"
OUT_PARQUET = "ncbi_latest_assembly_index.parquet"  # best for big data
OUT_CSV = "ncbi_latest_assembly_index.csv"  # optional
MAX_WORKERS = 32  # tune: 16-64
TIMEOUT = 60
RETRIES = 4
BACKOFF_BASE = 1.5
JITTER = 0.25

In [ ]:
session = requests.Session()
session.headers.update({"User-Agent": "genome-bert-indexer/1.0 (polite; contact: you@yourlab.edu)"})

In [16]:
def fetch_latest_listing(base_url: str):
    latest_url = urljoin(base_url, "latest_assembly_versions/")
    last_err = None

    for attempt in range(RETRIES):
        try:
            r = session.get(latest_url, timeout=TIMEOUT)
            # handle soft rate-limits
            if r.status_code in (429, 500, 502, 503, 504):
                raise RuntimeError(f"HTTP {r.status_code}")

            r.raise_for_status()
            soup = BeautifulSoup(r.text, "html.parser")
            pre = soup.find("pre")
            if pre is None:
                # sometimes NCBI pages differ, but usually pre exists
                return {"base_url": base_url, "latest_url": latest_url, "ok": False, "error": "no_pre_tag", "assemblies": []}

            # NCBI listing: <a href="GCA_xxx/">GCA_xxx/</a>
            assemblies = []
            for a in pre.find_all("a", href=True):
                txt = a.get_text(strip=True)
                if txt == "Parent Directory":
                    continue
                # directories usually end with /
                if not txt.endswith("/"):
                    continue
                assemblies.append(txt.rstrip("/"))

            return {"base_url": base_url, "latest_url": latest_url, "ok": True, "error": None, "assemblies": assemblies}

        except Exception as e:
            last_err = str(e)
            print(f"Error fetching {latest_url} (attempt {attempt + 1}/{RETRIES}): {last_err}")
            sleep_s = (BACKOFF_BASE**attempt) + random.random() * JITTER
            time.sleep(sleep_s)

    return {"base_url": base_url, "latest_url": latest_url, "ok": False, "error": last_err, "assemblies": []}

In [17]:
df = pd.read_csv(CSV_IN)
len(df.groupby("source_url"))

100096

In [ ]:
bases = df.loc[df["text"].eq("latest_assembly_versions/"), "source_url"].dropna().drop_duplicates().tolist()

In [3]:
df.drop(columns=["href", "abs_url", "name"], inplace=True)
df["name"] = df["source_url"].apply(lambda x: x.split("bacteria/")[-1].replace("/", ""))

In [4]:
df.text.unique()

array(['all_assembly_versions/', 'latest_assembly_versions/',
       'annotation_hashes.txt', 'assembly_summary.txt', 'reference/',
       'assembly_summary_historical.txt'], dtype=object)

In [5]:
no_latest_assembly_versions = df.groupby("source_url").filter(lambda g: "latest_assembly_versions/" not in set(g["text"]))

# Show summary
print("Filtered rows:", len(no_latest_assembly_versions))
print("Unique source_url count:", no_latest_assembly_versions["source_url"].nunique())

display(no_latest_assembly_versions.head())

Filtered rows: 1962
Unique source_url count: 1002


,source_url,text,is_dir,name
1196,https://ftp.ncbi.nlm.nih.gov/genomes/genbank/b...,all_assembly_versions/,True,Achromobacter_sp._79A6
1197,https://ftp.ncbi.nlm.nih.gov/genomes/genbank/b...,assembly_summary_historical.txt,False,Achromobacter_sp._79A6
1246,https://ftp.ncbi.nlm.nih.gov/genomes/genbank/b...,all_assembly_versions/,True,Achromobacter_sp._B4U2B
1247,https://ftp.ncbi.nlm.nih.gov/genomes/genbank/b...,assembly_summary_historical.txt,False,Achromobacter_sp._B4U2B
1446,https://ftp.ncbi.nlm.nih.gov/genomes/genbank/b...,all_assembly_versions/,True,Achromobacter_sp._T5-5


In [6]:
filtered_groups = df[df.text == "latest_assembly_versions/"]
print("Filtered rows:", len(filtered_groups))
filtered_groups.head()

Filtered rows: 99094


,source_url,text,is_dir,name
1,https://ftp.ncbi.nlm.nih.gov/genomes/genbank/b...,latest_assembly_versions/,True,Abditibacteriaceae_bacterium
5,https://ftp.ncbi.nlm.nih.gov/genomes/genbank/b...,latest_assembly_versions/,True,Abditibacterium_sp.
9,https://ftp.ncbi.nlm.nih.gov/genomes/genbank/b...,latest_assembly_versions/,True,Aaplasma_endosymbiont_of_Hyalomma_asiaticum
13,https://ftp.ncbi.nlm.nih.gov/genomes/genbank/b...,latest_assembly_versions/,True,Abditibacteriota_bacterium
17,https://ftp.ncbi.nlm.nih.gov/genomes/genbank/b...,latest_assembly_versions/,True,Abditibacterium_utsteinense


In [7]:
filtered_groups["source_url"].iloc[6]

'https://ftp.ncbi.nlm.nih.gov/genomes/genbank/bacteria/Abiotrophia_defectiva/'

In [12]:
# Get the HTML content from the first URL in filtered_groups
url = filtered_groups["source_url"].iloc[6] + "latest_assembly_versions/"
response = requests.get(url)
html_content = response.text

# Parse HTML with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

# Find all <a> links inside <pre> tag
pre_tag = soup.find("pre")
links = pre_tag.find_all("a") if pre_tag else []

# Extract href and text from each link
_df = pd.DataFrame([{"href": link.get("href"), "text": link.get_text()} for link in links if link.get_text() != "Parent Directory"])

_df

,href,text
0,GCA_000160075.2_ASM16007v2/,GCA_000160075.2_ASM16007v2/
1,GCA_013267415.1_ASM1326741v1/,GCA_013267415.1_ASM1326741v1/
2,GCA_015259635.1_ASM1525963v1/,GCA_015259635.1_ASM1525963v1/
3,GCA_026783725.1_ASM2678372v1/,GCA_026783725.1_ASM2678372v1/
4,GCA_037041345.1_ASM3704134v1/,GCA_037041345.1_ASM3704134v1/
5,GCA_042644865.1_ASM4264486v1/,GCA_042644865.1_ASM4264486v1/
6,GCA_053762285.1_ASM5376228v1/,GCA_053762285.1_ASM5376228v1/
7,GCA_905373775.1_SRR9217492-mag-bin.12/,GCA_905373775.1_SRR9217492-mag-bin.12/
8,GCA_916438635.1_DRR214962_bin.30_metaWRAP_v1.1...,GCA_916438635.1_DRR214962_bin.30_metaWRAP_v1.1...
9,GCA_937930255.1_ERR589701_bin.29_CONCOCT_v1.1_...,GCA_937930255.1_ERR589701_bin.29_CONCOCT_v1.1_...


In [10]:
links

[]